In [1]:
#old adaboost implementation library

In [ ]:
#make it so can be called without returning a list
#incorporate SMOTe and ADACost
#incroporate pruning
#make it kfold validate naturally
def Adaboost (T, data, label):
    N=data.shape[0]
    dim = data.shape[1]
    h = np.zeros([T, 3], dtype=np.float64)
    #print(h)
    alpha = np.zeros(T, dtype=np.float64)
    weight = ((pd.Series([1]*N))/N).values
    err = np.ones(T, dtype=np.float64) * np.inf
    ensemble_list = [] #to hold each ensemble member

    #0 - for each boosting round
    for t in range(T):
        #1 - iterate through every feature  
        for feature in range(dim):
            weighted_error = np.float64(0)

            #========================
            #2 - GENERATE A DECISION STUMP FOR A FEATURE
            #========================
            threshold, sign, weighted_error = calculate_decision_stump(data[:,feature], weight, label)
            #returns best_thresh, best_dir, min_error

            #========================
            #3 - KEEP TRACK OF THE FEATURE WITH THE LOWEST WEIGHTED ERROR
            #========================   

            if weighted_error < err[t] : 
                err[t] = weighted_error         
                h[t] = [threshold, feature, sign]

        #========================
        #4 - CALCULATE ALPHA FOR BOOSTING ROUND t
        #========================            
        alpha[t] = calculate_alpha(err[t])
        #returns alpha as a value

        #========================
        #5 - CLASSIFY ALL SAMPLES BASED ON THE SELECTED FEATURE FOR BOOSTING ROUND t
        #========================
        classification = classify_dataset_against_weak_classifier(data[:,h[t][1].astype(int)], h[t][0], h[t][2])
        #returns classification as array

        #========================
        #6 - UPDATE WEIGHTS BASED ON THE CORRECTNESS OF THE CLASSIFICATION
        #========================   
        weight = update_weights(weight, alpha[t], classification, label)
        #returns weight as an array

        #========================
        #7 - NORMALISE REASSIGNED WEIGHTS
        #========================  
        weight = normalise_weights(weight)
        #returns weight as array 

        #put the current model into a list for pruning
        cdf = pd.DataFrame(pd.DataFrame([[h[t][0],h[t][1],h[t][2],alpha[t],err[t]]], columns= ["threshold", "feature", "direction","alpha", "error"]))
        
        ensemble_list.append(predict_ensemble(cdf, data, label))
        #--------------------------------------------
        #BOOSTING ALGORITHM DONE, NOW CREATE OUTPUT DATAFRAMES
        #--------------------------------------------
    classifier_df = pd.DataFrame(h, columns = ["threshold", "feature", "direction"])
    classifier_df["alpha"] = alpha
    classifier_df["error"] = err

    training_set_df = predict_ensemble(classifier_df, data, label)
    
    print("Classification of ", T, " iterations complete")
    
    return classifier_df, training_set_df, ensemble_list

In [ ]:
def AdaCost (T, data, label, cost):
    N = data.shape[0]
    dim = data.shape[1]
    ensemble_list = []
    
    if cost is None:
        #print("COST IS NONE")
        cost = np.ones(N)
    else:
        cost = np.full(N, cost)
        
    misclassification_costs = np.zeros(N)   
    h = np.zeros([T, 3], dtype=np.float64)
    alpha = np.zeros(T, dtype=np.float64)
    weight = cost / np.sum(cost) #if cost is none, this will return a vector of 0.005, N tall
    err = np.ones(T, dtype=np.float64) * np.inf    
    
    #0 - for each boosting round
    for t in range(T):
        #1 - iterate through every feature  
        for feature in range(dim):
            weighted_error = np.float64(0)

            #========================
            #2 - GENERATE A DECISION STUMP FOR A FEATURE
            #========================
            threshold, sign, weighted_error = calculate_decision_stump(data[:,feature], weight, label)
            #returns best_thresh, best_dir, min_error

            #========================
            #3 - KEEP TRACK OF THE FEATURE WITH THE LOWEST WEIGHTED ERROR
            #========================   

            if weighted_error < err[t] : 
                err[t] = weighted_error         
                h[t] = [threshold, feature, sign]

        #========================
        #4 - CLASSIFY ALL SAMPLES BASED ON THE SELECTED FEATURE FOR BOOSTING ROUND t
        #========================
        classification = classify_dataset_against_weak_classifier(data[:,h[t][1].astype(int)], h[t][0], h[t][2])
        #returns classification as array
        
        #========================
        #5 - CALCULATE MISCLASSIFICATION COSTS
        #========================
        #update the weighted error based on new formula
        err[t], misclassification_costs = misclassification_cost(cost, weight, classification, label, misclassification_costs)
        #see if htere is any difference between this and the original, if not can drop step

        #========================
        #6 - CALCULATE ALPHA FOR BOOSTING ROUND t
        #========================            
        alpha[t] = calculate_alpha(err[t])
        #returns alpha as a value

        #========================
        #7 - CALCULATE WEIGHT UPDATE USING COSTS FOR ROUND t
        #========================  
        weight = ADACost_update(weight, cost, alpha[t], classification, label)
            
        #========================
        #7 - NORMALISE REASSIGNED WEIGHTS
        #========================  
        weight = normalise_weights(weight)
        #returns weight as array 

        #put the current model into a list for pruning
        cdf = pd.DataFrame(pd.DataFrame([[h[t][0],h[t][1],h[t][2],alpha[t],err[t]]], columns= ["threshold", "feature", "direction","alpha", "error"]))
        
        ensemble_list.append(predict_ensemble(cdf, data, label))
        #--------------------------------------------
        #BOOSTING ALGORITHM DONE, NOW CREATE OUTPUT DATAFRAMES
        #--------------------------------------------
    classifier_df = pd.DataFrame(h, columns = ["threshold", "feature", "direction"])
    classifier_df["alpha"] = alpha
    classifier_df["error"] = err

    training_set_df = predict_ensemble(classifier_df, data, label)
    
    print("Classification of ", T, " iterations complete")
    
    return classifier_df, training_set_df, ensemble_list

In [ ]:
def SMOTEBoost(T, data, label, n, k, w): #same as ADAboost with n and k being numbner of resmaples and neighbors for SMOTE
    N=data.shape[0]
    dim = data.shape[1]
    ensemble_list = []
    h = np.zeros([T, 3], dtype=np.float64)
    #print(h)
    alpha = np.zeros(T, dtype=np.float64)
    weight = ((pd.Series([1]*N))/N).values
    err = np.ones(T, dtype=np.float64) * np.inf
    
    #0 - for each boosting round
    for t in range(T):
        #1 - iterate through every feature  
        for feature in range(dim):
            weighted_error = np.float64(0)
            
            #========================
            #2 - GENERATE A DECISION STUMP FOR A FEATURE
            #========================
            threshold, sign, weighted_error = calculate_decision_stump(data[:,feature], weight, label)
            #returns best_thresh, best_dir, min_error

            #========================
            #3 - KEEP TRACK OF THE FEATURE WITH THE LOWEST WEIGHTED ERROR
            #========================   

            if weighted_error < err[t] : 
                err[t] = weighted_error         
                h[t] = [threshold, feature, sign]

        #========================
        #4 - CALCULATE ALPHA FOR BOOSTING ROUND t
        #========================            
        alpha[t] = calculate_alpha(err[t])
        #returns alpha as a value

        #========================
        #5 - CLASSIFY ALL SAMPLES BASED ON THE SELECTED FEATURE FOR BOOSTING ROUND t
        #========================
        classification = classify_dataset_against_weak_classifier(data[:,h[t][1].astype(int)], h[t][0], h[t][2])
        #returns classification as array

        #========================
        #6 - UPDATE WEIGHTS BASED ON THE CORRECTNESS OF THE CLASSIFICATION
        #========================   
        weight = update_weights(weight, alpha[t], classification, label)
        #returns weight as an array

        #========================
        #7 - NORMALISE REASSIGNED WEIGHTS
        #========================  
        weight = normalise_weights(weight)
        #returns weight as array 
        
        #put the current model into a list for pruning
        cdf = pd.DataFrame(pd.DataFrame([[h[t][0],h[t][1],h[t][2],alpha[t],err[t]]], columns= ["threshold", "feature", "direction","alpha", "error"]))
        
        ensemble_list.append(predict_ensemble(cdf, data, label))
        
        #========================
        #8 - SMOTE: if the n is set too high, the SMOTE algo will swap the minority class automatically
        #========================
        SMOTE_data, SMOTE_label = SMOTE(data, label, n, k)
        data = np.append(data, SMOTE_data, axis=0)
        label = np.append(label, SMOTE_label)
        weight = SMOTE_weights(weight, n, w)

        #--------------------------------------------
        #BOOSTING ALGORITHM DONE, NOW CREATE OUTPUT DATAFRAMES
        #--------------------------------------------
    classifier_df = pd.DataFrame(h, columns = ["threshold", "feature", "direction"])
    classifier_df["alpha"] = alpha
    classifier_df["error"] = err

    training_set_df = predict_ensemble(classifier_df, data, label)
    
    print("Classification of ", T, " iterations complete")
    
    return classifier_df, training_set_df, ensemble_list

In [ ]:
#prediction probabilities for ADAboost.

def ADA_pred_probs2(alphas, members):
    pred_array = []
    weighted_sum = np.zeros(len(members[0]))
    
    for i in range(len(train_list)):
        pred_array.append(train_list[i]['classification'])

    for i in range(len(alpha_list)):
         pred_array[i] = train_list[i]['classification'] * alpha_list[i]
    
    pred_df = pd.DataFrame(np.array(pred_array).T)
    
    for i in range(len(weighted_sum)):
        weighted_sum[i] += pred_df.iloc[i].sum()

    for i in range(len(weighted_sum)):
        weighted_sum[i] = sigmoid(-2 * weighted_sum[i])

    prob_class_1 = 1- weighted_sum
    prob_class_0 = weighted_sum
        
    return prob_class_1, prob_class_0

def ADA_pred_probs(alphas, members):
    pred_array = []
    weighted_sum = np.zeros(len(members[0]))

    for i in range(len(alpha_list)):
         pred_array.append(members[i]['classification'] * alphas[i])
    
    pred_array = (np.array(pred_array).T)

    for i in range(len(weighted_sum)):
        weighted_sum[i] = sigmoid(-2 * pred_array[i].sum())

    prob_class_pos = 1- weighted_sum
    prob_class_neg = weighted_sum
        
    return prob_class_pos, prob_class_neg

In [ ]:
def Siggy_stump(data, weights, label):
    data = pd.DataFrame(data)
    min_error = float('inf')
    best_thresh = 0
    best_dir = 0
    
    # Calculate thresholds
    feature_min = data.min().iloc[0]
    feature_max = data.max().iloc[0]
    #print("Min: ", feature_min, "max: ", feature_max)
    interval = (feature_max - feature_min) / 100.0
    #print("INTERVAL: ", interval)
    thresholds = pd.interval_range(start=feature_min-interval*2, end=feature_max+interval*2, freq=interval)  #optimise thresholds?
    #print("THRESHOLDS: ",type(thresholds[1]), "\n",  thresholds[1])
    
    for direction in [1, -1]:
        for thresh in thresholds:
            if direction == 1:
                threshold = sigmoid(thresh.left)
#                 print("THRESH: ", thresh)
                log_class = logistic_class(data, label, 1, 0.01, threshold, weights)
#                 print(f"LOG CLASS direction {direction} ", log_class)
                predictions = ( log_class >= threshold).astype(int) * 2 - 1 #this transforms the values from {0, 1} to {-1, 1}. Specifically, 1 remains 1, and 0 becomes -1.
#                 print(predictions)
            else:
                threshold = sigmoid(thresh.left)
                log_class = logistic_class(data, label, 1, 0.01, threshold, weights)
                #print(f"LOG CLASS direction {direction} ", log_class)
                predictions = (log_class < threshold).astype(int) * 2 - 1 #this transforms the values from {0, 1} to {-1, 1}. Specifically, 1 remains 1, and 0 becomes -1.
#                 print(predictions)
#         print("PREDICTIONS ", [thresh], ": ", predictions, """""")
            
            # Calculate misclassifications    
            #print(len(predictions), " : ", labels.shape)
            misclassifications = (predictions != labels)#.astype(int)
            #print("MISSCLAS: ", misclassifications, """""")
            
            # Calculate weighted error
            weighted_error = (misclassifications * weights).sum()
            
#             print("WEIGHTED_ERROR: ", weighted_error, """""")
            if weighted_error < min_error:     
                min_error = weighted_error
                best_thresh = thresh.left
                best_dir = direction    
    
    return best_thresh, best_dir, min_error

In [ ]:
def add_intercept(X): #to help plot
    intercept = np.zeros((X.shape[0], 1))
    return np.concatenate((intercept, X), axis=1)



def hypothesis(data, theta, bias): 
    return sigmoid(np.dot(theta, data.T)+bias) #need to output per data row

def cost(hypothesis, label): #error
    return (-label * np.log(hypothesis) - (1 - label) * np.log(1 - hypothesis)).mean()

def train_logistic(data, label, iterations, learning_rate, weight):
    #print(data.shape)
    theta = 0  #np.zeros(data.shape[1])
    bias = 0 #initialise 0 bias
    data = np.array(data)

    for i in range(iterations):
        H = hypothesis(data, theta, bias)
        #gradient_descent
        H = (H - label )* weight

        gradient_t = np.dot(data, (H - label)*weight) / len(label)
        #gradient_b = np.sum(H - label) / len(label)
        theta -= learning_rate * gradient_t
        theta = theta[0].sum()
    return theta, bias

def predict_logistic(data, theta, bias, threshold):
    predictions = hypothesis(data, theta, bias)
    H = predictions.copy()
    
    for i in range(len(predictions)):
        if predictions[i] >= threshold:
            predictions[i] = 1
        else:
            predictions[i] = -1
            
    return predictions, H

def logistic_class(data, label, iterations, learning_rate, threshold, weight):
    #data = add_intercept(data)
    theta, bias = train_logistic(data, label, iterations, learning_rate, weight)
    bias = 0
#     print("THETA:" , theta)
#     print("THRESHOLD:" , type(threshold) ,"\n", threshold)

    #result, H = predict_logistic(data, theta, bias, threshold)
    result = hypothesis(data, theta, bias)

    return result#, theta, bias, H

def Siggy_stump(data, weights, label):
    data = pd.DataFrame(data)
    min_error = float('inf')
    best_thresh = 0
    best_dir = 0
    
    # Calculate thresholds
    feature_min = data.min().iloc[0]
    feature_max = data.max().iloc[0]
#     print("Min: ", feature_min, "max: ", feature_max)
#     nterval = (feature_max - feature_min) / 100.0
    
    thresholds = [random.uniform(feature_min, feature_max) for _ in range(10)]
    
    #thresholds = pd.interval_range(start=feature_min-interval*2, end=feature_max+interval*2, freq=interval)  #optimise thresholds?
    #print("THRESHOLDS: ",type(thresholds[1]), "\n",  thresholds[1])
    
    for i in r
    
    for thresh in thresholds:
        predictions = []
        log_class = logistic_class(data, label, 1, 0.01, thresh, weights)
        log_class = np.where(log_class >= thresh, 1, -1)

        predictions = log_class    
#         print(predictions)

        misclassifications = (predictions != labels)#.astype(int)
        #print("MISSCLAS: ", misclassifications, """""")

        # Calculate weighted error
        weighted_error = (misclassifications * weights).sum()

#             print("WEIGHTED_ERROR: ", weighted_error, """""")
        if weighted_error < min_error:     
            min_error = weighted_error
            best_thresh = thresh
            best_dir = 1    
    
    return best_thresh, best_dir, min_error
    
#best threshold becomes what... the decision boundary of the algo?
#if bias != 0 the prediction resukts in all 1 or all -1, will need to force bias to be 0
#get H(x) for each sample, and if it was classified wrongly, push it towards the right direction. I.e. if it shuold be 1, but was -1, make it wighted higher. 

In [ ]:
def AdaboostKNN (T, data, label, k):
    N=200
    h = pd.DataFrame()
    #print(h)
    alpha = np.zeros(T, dtype=np.float64)
    weight = ((pd.Series([1]*N))/N).values
    err = np.ones(T, dtype=np.float64) * np.inf
    
    #0 - for each boosting round
    for t in range(T):
        weighted_error = np.float64(0)

        #========================
        #2 - GENERATE KNN with K = 1
        #========================
        weighted_error, predictions = calc_knn(data, weight, label, k)
        #returns min err
        #print(feature, " : : " ,weighted_error)
        #========================
        #3 - KEEP TRACK OF THE FEATURE WITH THE LOWEST WEIGHTED ERROR
        #========================   
        
        if weighted_error < err[t] : 
            err[t] = weighted_error         
            h[t] = predictions
        #print(h)
        #========================
        #4 - CALCULATE ALPHA FOR BOOSTING ROUND t
        #========================            
        alpha[t] = calculate_alpha(err[t])
        #returns alpha as a value

        #========================
        #5 - CLASSIFY ALL SAMPLES BASED ON THE SELECTED FEATURE FOR BOOSTING ROUND t
        #========================
        #print("H: ", h[t][0])
        classification = classify_knn(data, label)
        #returns classification as array
        #print("CLASSIFICATION: ", classification)
        #========================
        #6 - UPDATE WEIGHTS BASED ON THE CORRECTNESS OF THE CLASSIFICATION
        #========================   
        weight = update_weights(weight, alpha[t], classification, label)
        #returns weight as an array

        #========================
        #7 - NORMALISE REASSIGNED WEIGHTS
        #========================  
        weight = normalise_weights(weight)
        #returns weight as array 
        print("WEIGHTS UPDATED: ", weight)
        #--------------------------------------------
        #BOOSTING ALGORITHM DONE, NOW CREATE OUTPUT DATAFRAMES
        #--------------------------------------------
        
    #mode h along the rows to get average vote
    #classifier_df = pd.DataFrame(h, columns = ["feature"])
    classifier_df = pd.DataFrame(alpha, columns = ["alpha"])
    #classifier_df["alpha"] = alpha
        
    training_set_df = pd.DataFrame(data)
    training_set_df['prediction'] = h.mode(axis=1)
    training_set_df['label'] = label
    #training_set_df['sum_alpha'] = 0 
    #training_set_df['pos_votes'] = 0 
    #training_set_df['neg_votes'] = 0 

    #training_set_df = sum_classifier_votes_for_each_sample(data, training_set_df, classifier_df)
    #total_alpha_votes = np.sum(classifier_df.alpha)
    #training_set_df['classification'] = 0
    #training_set_df['margin'] = 0
    #training_set_df['total_alpha_votes'] = total_alpha_votes
    #result = training_set_df[['sum_alpha','pos_votes','neg_votes','total_alpha_votes']].apply(lambda data: margin_calculation(*data), axis=1)
    ##training_set_df['margin'] = result.apply(lambda data: data[0])
    #training_set_df['classification'] = result.apply(lambda data: data[1])
    #training_set_df['true_class_label'] = label
    #training_set_df['sign_of_margin'] = training_set_df[['margin', 'classification', 'true_class_label']].apply(lambda data: sign_of_margin(*data), axis=1)

    classifier_df["error"] = err
    
    return classifier_df, training_set_df 



In [ ]:
def knn_weights(k, distances):
    result = np.zeros(k, dtype = np.float32)
    sum = 0.0
    for i in range(k):
        result[i] += 1.0 / distances[i]
        sum += result[i]
    result /= sum
    
    return result

def get_neighbours(data, row, label, k): 
    distances = []
    
    for i in data:
        dist = euclidean_dist(row, i)
        distances.append(dist)
       
    data_df = pd.DataFrame(zip(data[:,0], data[:,1], distances, label), columns = ['0', '1', 'distance', 'label'])
    data_df = data_df.sort_values(by = 'distance')
    data_df = data_df[1:] #drop the row representing the input row
    #print(data_df)
    
    weights = knn_weights(k, np.array(data_df['distance']))
    neighbours = data_df.head(k)
    neighbours['weights'] = weights
        
    return neighbours

def knn_predict_class(train, test_row, label, k):
    neighbours = get_neighbours(train, test_row, label, k)
    
    pc = neighbours.groupby('label')['weights'].sum()
    
    #if neighbours['label'].mode()[0] == 1:
    #    return 1
    #else: 
    #    return -1
    
    return pc.idxmax()
    
def classify_knn(data, label):
    classification = np.zeros(len(data))
    
    for i in range(len(data)):
        classification[i] = knn_predict_class(data, i, label, 1)
    return classification


In [ ]:
def calc_knn(data, weight, label, k):
    """
    Calculate the knn given feature.
    
    Args:
    data : Dataset containing just as single feature
    weights: Sample weights
    labels: Sample class labels
    
    Returns:
    tuple: (best_threshold, best_direction, min_error)
    """
    #data = pd.DataFrame(data)
    min_error = float('inf')
    best_thresh = 0
    best_dir = 0
    
    predictions = []
    
    for row in data:
        predictions.append(knn_predict_class(data, row, label, k))
        
    #print(predictions)
       
    # Calculate misclassifications          
    misclassifications = (label != predictions).astype(int)
    print("MISSCLAS: ", misclassifications, """""")
    # Calculate weighted error
    weighted_error = (misclassifications * weight).sum()
    
    
    print("WEIGHTED_ERROR: ", weighted_error, """""")
    if weighted_error < min_error:     
        min_error = weighted_error
        #best_thresh = thresh.left
        #best_dir = direction    
    
    return min_error, predictions #best_thresh, best_dir

In [ ]:
def find_threshold(data, label, theta, bias, weights, thresholds): #dependings on the data set, this will optimise to all -1 or all 1
    weighted_error = np.float64(0)
    min_error = float('inf')
    best_thresh = 0
    best_dir = 0
    
    for thresh in thresholds:
        predictions = predict_logistic(data, theta, bias, thresh)
        print("threshold: ", thresh, " : ", predictions)    
        misclassifications = (predictions != label).astype(int)

        weighted_error = (misclassifications * weights).sum()
        if weighted_error < min_error:
                min_error = weighted_error
                best_thresh = thresh
                #best_dir = direction
                best_pred = predictions.copy()
                
    return best_thresh, min_error, best_pred

#use weights and return optimal weighting?